## 5. Urban-Rural Polygon Generation from GHS-SMOD
This notebook processes the Global Human Settlement Layer (GHS-SMOD) raster data. Its primary function is to generate vector polygons that define different levels of urbanization. These polygons are saved and used as an input for the next notebook (6_urban_rural_segregation) to efficiently classify building footprints. The main input, the HGS-SMOD layer needs to be download from [here](https://human-settlement.emergency.copernicus.eu/download.php?ds=smod)

### Initial configuration

    config_str = """{"REGION": "Patan",
    "COS_ENDPOINT_URL": "https://s3.direct.eu-de.cloud-object-storage.appdomain.cloud",
    "COUNTRY_NAME": "India",
    "FILE_PREFIX": "Patan",
    "VIDA_PARQUET_BUCKET": "parquets",
    "VIDA_S2_PARTITIONS": "onboarding-bucket-j",
    "COS_AUTH_ENDPOINT_URL":"https://iam.cloud.ibm.com/oidc/token",
    "COS_APIKEY": "xxx",
    "UTILS_BUCKET": "notebook-utils-bucket",
    "VIDA_COUNTRIES_BUILDINGS": "onboarding-bucket-j"}"""

* `REGION`: Region of study.
* `COS_ENDPOINT_URL`: The S3 API endpoint for your IBM COS instance.
* `COUNTRY_NAME`: Defines the target country to filter the google dataset.
* `VIDA_PARQUET_BUCKET`: The name of the source bucket where the Parquet files are stored.
* `COS_AUTH_ENDPOINT_URL`: The authentication endpoint for generating IBM IAM tokens.
* `COS_APIKEY`: Your unique IBM Cloud API key for programmatic access.
* `UTILS_BUCKET`: Where the .py file to download the package `filtering_grid_generator`.
* `VIDA_COUNTRIES_BUILDINGS`: Bucket to upload results.

In [1]:
import getpass
import json
import pandas as pd
import geopandas as gpd
import jaydebeapi as jdbc
import jpype
import os
import shapely
import rasterio as rio
from rasterio.plot import show
from rasterio.mask import mask
import matplotlib.pyplot as plt
import  rioxarray
from skimage import measure as M
import requests
from botocore.client import Config
import ibm_boto3
import io
from datetime import datetime
from collections import Counter

In [3]:
datetime.now().strftime("%Y-%m-%d %H:%M:%S")

'2025-10-03 13:13:36'

In [4]:
config = json.loads(config_str)

The next cell ask for two paths. The first one "administrative boundaries" refers to the boundaries of the selected area where the script will focus to obtain the building footprints. The second one, the geotif file from the Globam Human Settlement website.

In [5]:
input_file = "/home/julian/notebooks/handover/notebooks/district_boundaries/Patan.geojson"
in_tiff_file = "/home/julian/notebooks/GHS_SMOD_E2030_GLOBE_R2023A_54009_1000_V2_0_R7_C25.tif"
geojson_read = gpd.read_file(input_file)

In [6]:
regions_polygons = {
    config["REGION"]: 
        [geojson_read.explode().geometry.iloc[0]]
        
}
regions_polygons

{'Patan': [<POLYGON ((72.11 24.045, 72.113 24.044, 72.119 24.041, 72.124 24.039, 72.133...>]}

This script acts as a configuration helper to define two different classification schemes for urban-rural segregation based on the GHS-SMOD numerical codes.

In [7]:
#-----------------------------------------------------------------------------------------------------------------------------
# New/Editted code 
# Function to set segregation depending on style
def set_segregation(style):
    if style == "overview":
        segregation = {
            'URBAN': [22, 23, 30],
            'SUBURBAN': [21],
            'RURAL': [12, 13],
        }
        segregation_priorities = ['URBAN', 'SUBURBAN']

    elif style == "detailed":
        segregation = {
            'URBAN_CENTER': [30],
            'DENSE_URBAN': [23],
            'SEMI_DENSE_URBAN': [22],
            'SUBURBAN_PERI_URBAN': [21],
            'RURAL_CLUSTER': [13],
            'LOW_DENSITY_RURAL': [12],
        }
        segregation_priorities = ['URBAN_CENTER', 'DENSE_URBAN', 'SEMI_DENSE_URBAN', 'SUBURBAN_PERI_URBAN', 'RURAL_CLUSTER', 'LOW_DENSITY_RURAL']

    else:
        raise ValueError("Unknown segregation style")

    return segregation, segregation_priorities

#Reproject tiff to EPSG:4326 CRS uring Rasterio
def reproject_tif_CRS(filename: str):
    rds = rioxarray.open_rasterio(filename)
    rds_4326 = rds.rio.reproject("EPSG:4326")
    rds_4326.rio.to_raster(filename, compress="DEFLATE")

def get_region_smod_tiff(in_tiff_file, out_tiff_file, polygon):

    with rio.open(in_tiff_file) as src:
        #mask the raster data based on geometry
        out_image, out_transform = mask(src, [polygon], crop=True)
        out_meta = src.meta
        
        #update metadata
        out_meta.update(
            {
                "height": out_image.shape[1],
                "width": out_image.shape[2],
                "transform": out_transform,
                "nodata": -1.0
            }
        )
        #Save the masked taster data to a new tiff file
        with rio.open(out_tiff_file, "w", **out_meta) as dest:
            dest.write(out_image)

from collections import Counter

def generate_segregated_geojson(in_tiff_file, out_json_file, segregation, segregation_priorities):
    try:
        with rio.open(in_tiff_file) as src:
            
            segregated_polygons = {}
            #Loop through each and segregate polygons
            for k, v in segregation.items():
                data = src.read(1)
                #Update values in the raster data based on the defined range
                data[data > max(v)] = 0
                data[(data < min(v)) & (data > 0)] = 0

                for layer in v:
                    data[data == layer] = max(v)

                flatten = data.flatten(order='C')

                #print(list(set(flatten)))
                #print(Counter(flatten))

                contours = M.find_contours(data, max(v)/2)

                print(f'{len(contours)} contours {max(v)} find for {k}')
                polygons = []
                # Convert pixel coords to latlon coords and create shapely polygons
                for contour in contours:

                    latlon_coords = []

                    for coords in contour:
                        lon, lat = rio.transform.xy(src.transform, coords[0], coords[1])
                        latlon_coords.append([lon, lat])

                    latlon_coords.append(latlon_coords[0])
                    latlon_coords = shapely.Polygon(latlon_coords)
                    
                    polygons.append(latlon_coords)

                segregated_polygons[k] = polygons
                
            # print(f"Cloud polygond were found: {len(polygons)}")

        
    except Exception as e:
        print(f"Exception occured: {e}")
        
        
    #Initialize an empty GeoJSON object
    geojson = {
    "type": "FeatureCollection",
    "features": []
    }

    for area in segregation_priorities:

        polygons = []

        #Retrieve the polygons corresponding to area
        poly_temp = segregated_polygons[area]

        flag = True
        #Iterate through each polygon in the list
        for idx, p1 in enumerate(poly_temp):

            flag = True
            #Check for intersections with subsequent polygons
            for idxy, p2 in enumerate(poly_temp[idx + 1: ]):

                if p1.intersects(p2):
                    #If intersection occurs merge the polygons
                    poly_temp[idx + idxy + 1] = p2.union(p1)
                    flag = False
                    break
            # If no intersection occured, add the polygon to the list
            if flag: polygons.append(p1)

        for coords in polygons:
            #Create geojson feature for each polygon
            feature = {
                "type": "Feature",
                "properties": {'seg_type': area},
                "geometry": {
                    "coordinates": json.loads(shapely.to_geojson(coords))['coordinates'],
                    "type": "Polygon"
                    }}
                
            geojson['features'].append(feature)
        
        #Write the cleaned geojson to json file
    with open(out_json_file, "w") as outfile: 
        json.dump(geojson, outfile)


In [8]:
# init S3 client in order to work with last tiff file version
cos_client = ibm_boto3.client(service_name='s3',
                              ibm_api_key_id=config["COS_APIKEY"],
                              config=Config(signature_version='oauth'),
                              endpoint_url=config["COS_ENDPOINT_URL"])


# import external utils library
response = cos_client.list_objects_v2(Bucket=config["UTILS_BUCKET"])

utils_to_download = ['india_state.geojson']

try:
    for obj in response['Contents']:
        name = obj['Key']
        if name in utils_to_download:
            streaming_body_1 = cos_client.get_object(Bucket=config["UTILS_BUCKET"], Key=name)['Body']
            print("Copying to localStorage :  " + name)
            with io.FileIO(name, 'w') as file:
                for i in io.BytesIO(streaming_body_1.read()):
                    file.write(i)
                
    print('External utils succesfully imported')
except Exception as e:
    print('Error occured: ', e)

External utils succesfully imported


This script follows a two-phase workflow to process a large GeoTIFF file. First, it clips the raster into smaller, region-specific files. Second, it analyzes each of these smaller files to generate urban segregation GeoJSONs.

In [9]:
reproject_tif_CRS(in_tiff_file)

tiff_filenames = []
for region_name, region_polygons in regions_polygons.items():
    
    for pidx, polygon in enumerate(region_polygons):
        
        out_tiff_file = f"{region_name.replace(' ', '_')}_{pidx}.tif"
        get_region_smod_tiff(in_tiff_file, out_tiff_file, polygon)
        
        tiff_filenames.append(out_tiff_file)

for style in ["overview", "detailed"]:
    segregation, segregation_priorities = set_segregation(style)

    for tif_name in tiff_filenames:
        print(f'Processing {tif_name}')
        json_filename = tif_name.replace('.tif', f'_{style}.json')
        generate_segregated_geojson(tif_name, json_filename, segregation, segregation_priorities)   

Processing Patan_0.tif
11 contours 30 find for URBAN
41 contours 21 find for SUBURBAN
65 contours 13 find for RURAL
Processing Patan_0.tif
1 contours 30 find for URBAN_CENTER
1 contours 23 find for DENSE_URBAN
9 contours 22 find for SEMI_DENSE_URBAN
41 contours 21 find for SUBURBAN_PERI_URBAN
36 contours 13 find for RURAL_CLUSTER
84 contours 12 find for LOW_DENSITY_RURAL


The next cells automates the process of merging, filtering, and uploading GeoJSON files. It operates in two distinct modes—"overview" and "detailed"—to create separate aggregated files for each style. Then it uploads the files to the specified bucket.

In [10]:
#----------------------------------------------------------------------------------------------------------------
# new code
jsons = [i for i in os.listdir(os.getcwd()) if '.json' in i]

# Run merge twice: once for overview, once for detailed
for style in ["overview", "detailed"]:
    segregation, segregation_priorities = set_segregation(style)

    for region_name, region_polygons in regions_polygons.items():
        region_name = region_name.replace(' ', '_')

         # Only pick JSON files for the current style
        region_jsons = [i for i in jsons if region_name in i and i.endswith(f"_{style}.json")]
        print(f"{region_name} ({style}), {region_jsons}")

        all_features = []
        if len(region_jsons) > 1:
            for region_json in region_jsons:
                geojson = json.load(open(region_json))
                if "features" in geojson:
                    
                    valid_categories = segregation.keys() 
                    
                    for feature in geojson["features"]:
                        seg_type = feature["properties"]["seg_type"]
                        if seg_type in valid_categories:    # filter using the active dict
                            all_features.append(feature)

            out_geojson = {
                'type': 'FeatureCollection',
                'features': all_features
            }

            out_json_file = f"{region_name}_{style}.json"
            with open(out_json_file, "w") as outfile:
                json.dump(out_geojson, outfile)


        
        else:
            out_json_file = f"{region_name}_{style}.json"
            with open(region_jsons[0], "r") as infile:
                data = json.load(infile)
                with open(out_json_file, "w") as outfile:
                    json.dump(data, outfile)
                    print(f"Saved {out_json_file}")
                    

        # upload file to the COS bucket
        if type(config["OUTPUT_BUCKET"]) == str:
            try:
                cos_client.upload_file(
                    Filename=out_json_file,
                    Bucket=config["OUTPUT_BUCKET"],
                    Key=out_json_file,
                    ExtraArgs={'ContentDisposition': 'attachment'}
                )
                print(f'File {out_json_file} successfully uploaded to the COS {config["OUTPUT_BUCKET"]} bucket')
            except Exception as e:
                print(f'\033[91mFailed upload file to the bucket {config["OUTPUT_BUCKET"]}. Error: {e}')



Patan (overview), ['Patan_0_overview.json', 'Patan_overview.json']
File Patan_overview.json successfully uploaded to the COS onboarding-bucket-j bucket
Patan (detailed), ['Patan_0_detailed.json', 'Patan_detailed.json']
File Patan_detailed.json successfully uploaded to the COS onboarding-bucket-j bucket


In [11]:
datetime.now().strftime("%Y-%m-%d %H:%M:%S")

'2025-10-03 13:13:37'